# 07: Candidate explanations on development models

Design decisions
- Models: the development XGBoost models of notebook 06 for tasks A and B, at the plan default (100 trees of depth 4) and regularised (min_child_weight 50, gamma 2, max_bin 32); their hashes are asserted against results/06_encoding_tests.json.
- Explained patients: a seeded random sample of dev-cal, 60 patients per task for the default model and 300 for the regularised one (the default model costs about 50 times more per patient).
- Candidate sets (Algorithm 3): M = 10 feature orders (order 1 frees the least-used features first, orders 2 to 10 are seeded permutations); a deletion-based AXp per order; each AXp inflated feature by feature, each side by binary search, keeping every sound box visited. Soundness: interval pre-check, then the exact leaf branch-and-bound. Candidate sets are cached on Drive by classifier hash and settings.
- Every distinct candidate box is re-checked by the second exact procedure (real thresholds and fractions); a disagreement stops the notebook, a box beyond the search budget is removed. Z3 checks a random sample of boxes per model (10 s timeout per box, at most 60 boxes or 150 s per model).
- Reported per patient: seconds, number of candidates, size of the smallest candidate and of the first AXp, and dev-cal support (number of dev-cal patients inside the box; labels are not used). As a projection for notebook 05, support is scaled to calibration split 1 (three times dev-cal) and compared with the smallest support at which a box with zero errors could pass its class target under pi_Occ with delta_1 = 0.025 (alpha_0 = 0.05, alpha_1 = 0.70); this is a necessary condition, not a certification rate.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, write_json
from scax.data import public, splits
from scax.models.export import Ensemble
from scax.models.predictor import ExactPredictor
from scax.explain.candidates import CandidateGenerator
from scax.explain.verify import verify_boxes, z3_sample, drop_unverified
from scax.certify.prior import OccamPrior
from scax.certify.certificate import membership

SEED = 42
seed_everything(SEED)
M = 10
TASKS = ['brfss2015_heart', 'cdc_diabetes']
EXPLAIN = {'xgboost': 60, 'xgboost_reg': 300}
DELTA1, ALPHA = 0.025, {0: 0.05, 1: 0.70}
MODEL_DIR = paths.DERIVED / 'models'
CACHE_DIR = paths.DERIVED / 'cache'; CACHE_DIR.mkdir(parents=True, exist_ok=True)
REC06 = json.load(open(f'{paths.RESULTS}/06_encoding_tests.json'))
assert REC06['gate_G2']['passed']

## Candidate sets

In [ ]:
PATIENT_ROWS, BOX_SETS, GEN = [], {}, {}
rng = np.random.default_rng(SEED)
for task in TASKS:
    X, y = public.LOADERS[task](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    dc = splits.indices(code, 'dev_cal')
    Xdc = X.to_numpy(dtype=float)[dc]
    for name, n_pat in EXPLAIN.items():
        ens = Ensemble.load(MODEL_DIR / f'{task}__{name}.json')
        assert ens.hash() == REC06['models'][f'{task}__{name}']['sha256'], 'model differs from notebook 06'
        pred = ExactPredictor(ens); g = pred.grid
        gen = CandidateGenerator(pred, M=M, seed=SEED)
        cache_path = CACHE_DIR / f'07_{task}__{name}.pkl'
        gen.load(cache_path)
        pick = rng.choice(len(dc), n_pat, replace=False)
        Cp = g.cells(Xdc[pick]); yh = pred.predict_cells(Cp)
        t0 = time.time()
        for i in range(n_pat):
            gen.candidates(Cp[i], int(yh[i]))
        gen.save(cache_path)
        print(f'{task} {name}: {n_pat} patients, {time.time() - t0:.0f}s')
        GEN[(task, name)] = (gen, g, pred, Cp, yh, g.cells(Xdc))

## Verification of every candidate box

In [ ]:
VERIFY = []
for (task, name), (gen, g, pred, Cp, yh, Cdc) in GEN.items():
    keys = {gen.cell_key(c) for c in Cp}
    boxes = sorted({(c.key, c.y) for k in keys for c in gen.cache[k]})
    t0 = time.time()
    verdicts, times = verify_boxes(g, boxes)
    sub = {k: gen.cache[k] for k in keys}
    removed = drop_unverified(sub, verdicts)
    gen.cache.update(sub)
    z3res, zt = z3_sample(g, [b for b in boxes if verdicts[b] == 'sound'], 60, SEED, timeout_ms=10000, time_budget_s=150)
    VERIFY.append({'task': task, 'model': name, 'distinct_boxes': len(boxes), 'second_check_sound': sum(v == 'sound' for v in verdicts.values()),
                   'second_check_unknown': sum(v == 'unknown' for v in verdicts.values()), 'removed_from_candidate_sets': removed,
                   'second_check_median_ms': 1000 * float(np.median(times)), 'z3_sample_sound': z3res['sound'],
                   'z3_sample_unknown': z3res['unknown'], 'z3_sample_checked': z3res['checked'], 'z3_median_ms': 1000 * float(np.median(zt)) if zt else None})
VER = pd.DataFrame(VERIFY)
print(VER.round(1).to_string(index=False))

## Explanation size, support and cost

In [ ]:
def zero_error_support(log_pi, alpha):
    """Smallest n with (1 - alpha)^n <= delta_1 * pi: support a box needs even with no errors."""
    return np.ceil((np.log(DELTA1) + log_pi) / np.log1p(-alpha))

for (task, name), (gen, g, pred, Cp, yh, Cdc) in GEN.items():
    occ = OccamPrior(g.K)
    for i in range(len(Cp)):
        key = gen.cell_key(Cp[i])
        cands = gen.cache[key]
        if not cands:
            PATIENT_ROWS.append({'task': task, 'model': name, 'predicted': int(yh[i]), 'candidates': 0}); continue
        keys = [c.key for c in cands]
        support = np.asarray(membership(g, keys, Cdc).sum(axis=1)).ravel()
        log_pi = np.array([occ.log_pi(k) for k in keys])
        need = zero_error_support(log_pi, ALPHA[int(yh[i])])
        sizes = np.array([c.size for c in cands])
        smallest = int(np.argmin(sizes))
        first_axp = next((c for c in cands if c.kind == 'axp'), None)
        PATIENT_ROWS.append({'task': task, 'model': name, 'predicted': int(yh[i]), 'candidates': len(cands),
                             'seconds': gen.seconds.get(key), 'smallest_size': int(sizes.min()), 'first_axp_size': first_axp.size if first_axp else None,
                             'smallest_box_devcal_support': int(support[smallest]), 'max_devcal_support': int(support.max()),
                             'any_candidate_meets_zero_error_support': bool(np.any(3 * support >= need)),
                             'smallest_box_zero_error_support_needed': float(need[smallest])})
PAT = pd.DataFrame(PATIENT_ROWS)
PAT.to_csv(f'{paths.RESULTS}/07_candidate_stats.csv', index=False)
SUM = PAT.groupby(['task', 'model']).agg(
    patients=('candidates', 'size'), seconds_median=('seconds', 'median'),
    seconds_q1=('seconds', lambda s: s.quantile(0.25)), seconds_q3=('seconds', lambda s: s.quantile(0.75)),
    seconds_p95=('seconds', lambda s: s.quantile(0.95)), candidates_median=('candidates', 'median'),
    smallest_size_median=('smallest_size', 'median'), first_axp_size_median=('first_axp_size', 'median'),
    share_smallest_size_at_most_3=('smallest_size', lambda s: float((s <= 3).mean())),
    smallest_box_support_median=('smallest_box_devcal_support', 'median'),
    share_meeting_zero_error_support=('any_candidate_meets_zero_error_support', 'mean'),
    predicted_positive_share=('predicted', 'mean')).reset_index()
SUM = SUM.merge(VER, on=['task', 'model'])
SUM.to_csv(f'{paths.RESULTS}/07_summary.csv', index=False)
print(SUM.round(3).T.to_string())

In [ ]:
oracle = {f'{t}__{n}': {k: (v if not isinstance(v, list) else len(v)) for k, v in vars(GEN[(t, n)][0].oracle.stats).items()}
          for (t, n) in GEN}
write_json(f'{paths.RESULTS}/07_summary.json', {'seed': SEED, 'M': M, 'explained_patients': EXPLAIN, 'delta1': DELTA1,
                                               'alpha': ALPHA, 'summary': SUM.to_dict('records'), 'oracle_stats': oracle})
print(json.dumps(oracle, indent=1))

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "07: candidate explanations on development models for tasks A and B (default and regularised XGBoost), second-check verification of every box, size, support and cost"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)